(legacy inclusion) This notebook creates league-wide reference datasets for comparison in future stadium-specific analysis:

Includes:
1. every game from 2021-2025 with weather data, tagged by home team
2. seasonal weather baseline (avg across all games each day)
3. per-team home game averages vs league average

(Athletics are excluded from this analysis)

All outputs are saved to `Stadium Datasets/Final Datasets/`

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 35)

# === CONFIGURATION ===
SEASON_START = 2021
SEASON_END = 2025
SEASONS = range(SEASON_START, SEASON_END + 1)

# Exclusions
EXCLUDE = [('ATH', 2025)]  # Oakland A's 2025 (team relocated)

# Paths
DATASETS_DIR = os.path.join('..', 'Stadium Datasets', 'Final Datasets')
MASTER_FILE = os.path.join(DATASETS_DIR, 'master_data.csv')

# Output files
OUT_LEAGUE_WEATHER = os.path.join(DATASETS_DIR, 'league_weather_2021_2025.csv')
OUT_DAILY_WEATHER = os.path.join(DATASETS_DIR, 'league_daily_weather_2021_2025.csv')
OUT_BATTING_AVGS = os.path.join(DATASETS_DIR, 'league_batting_avgs_2021_2025.csv')

print(f"Seasons: {list(SEASONS)}")
print(f"Exclusions: {EXCLUDE}")
print(f"Datasets directory: {os.path.abspath(DATASETS_DIR)}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Seasons: [2021, 2022, 2023, 2024, 2025]
Exclusions: [('ATH', 2025)]
Datasets directory: /Users/avabrown/Desktop/DATASCI 192A/Stadium Analysis/Stadium Datasets/Final Datasets


## Loading Team Datasets

Read every `*_data_*.csv` file from Final Datasets and combine into one DataFrame. Join with `master_data.csv` on `game_pk` to get the correct `home_team` code.

In [2]:
# Load master data for home_team mapping
master = pd.read_csv(MASTER_FILE)
home_team_map = master[['game_pk', 'home_team']].drop_duplicates()
print(f"Master data: {len(master)} games, {master['home_team'].nunique()} teams")

# Find all team CSV files (exclude master_data, checkpoint, and league output files)
all_csvs = glob.glob(os.path.join(DATASETS_DIR, '*_data_*.csv'))
team_csvs = [f for f in all_csvs if not any(x in os.path.basename(f) for x in ['master_data', 'games_checkpoint', 'league_'])]
print(f"\nFound {len(team_csvs)} team CSV files:")
for f in sorted(team_csvs):
    print(f"  {os.path.basename(f)}")

Master data: 25155 games, 30 teams

Found 30 team CSV files:
  angels_data_2018.csv
  astros_data_2017.csv
  athletics_data_2015.csv
  blue_jays_data_2015.csv
  braves_data_2015.csv
  brewers_data_2015.csv
  cardinals_data_2015.csv
  cubs_data_2015.csv
  dbacks_data_2015.csv
  dodgers_data_2015.csv
  fenway_data_2016.csv
  giants_data_2020.csv
  guardians_data_2015.csv
  mariners_data_2015.csv
  marlins_data_2020.csv
  mets_data_2015.csv
  nationals_data_2015.csv
  orioles_data_2015.csv
  padres_data_2015.csv
  phillies_data_2015.csv
  pirates_data_2015.csv
  rangers_data_2015.csv
  rays_data_2015.csv
  reds_data_2015.csv
  rockies_data_2016.csv
  royals_data_2015.csv
  tigers_data_2015.csv
  twins_data_2015.csv
  white_sox_data_2015.csv
  yankee_data_2015.csv


In [ ]:
# Read and combine all team CSVs
frames = []
for csv_path in sorted(team_csvs):
    df = pd.read_csv(csv_path)
    frames.append(df)
    
all_games = pd.concat(frames, ignore_index=True)
print(f"Combined: {len(all_games)} total game records across {len(team_csvs)} team files")

all_games = all_games.merge(home_team_map, on='game_pk', how='left')

unmatched = all_games['home_team'].isna().sum()
if unmatched > 0:
    print(f"WARNING: {unmatched} games could not be matched to a home_team")
else:
    print(f"All games matched to home_team codes")

print(f"\nTeams present: {sorted(all_games['home_team'].unique())}")
print(f"Seasons present: {sorted(all_games['season'].unique())}")

Combined: 23700 total game records across 30 team files
All games matched to home_team codes

Teams present: ['ATH', 'ATL', 'AZ', 'BAL', 'BOS', 'CHC', 'CIN', 'CLE', 'COL', 'CWS', 'DET', 'HOU', 'KC', 'LAA', 'LAD', 'MIA', 'MIL', 'MIN', 'NYM', 'NYY', 'PHI', 'PIT', 'SD', 'SEA', 'SF', 'STL', 'TB', 'TEX', 'TOR', 'WSH']
Seasons present: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
# Filtering to only pre-defined filters (currently 2021-2025)
league = all_games[all_games['season'].isin(SEASONS)].copy()
print(f"After filtering to {SEASON_START}-{SEASON_END}: {len(league)} games")

# Apply exclusions
for team, year in EXCLUDE:
    before = len(league)
    league = league[~((league['home_team'] == team) & (league['season'] == year))]
    removed = before - len(league)
    print(f"Excluded {team} {year}: removed {removed} games")

print(f"\nFinal league dataset: {len(league)} games")
print(f"Teams: {league['home_team'].nunique()}")
print(f"\nGames per team:")
print(league.groupby('home_team')['game_pk'].count().sort_values(ascending=False).to_string())

After filtering to 2021-2025: 12058 games
Excluded ATH 2025: removed 0 games

Final league dataset: 12058 games
Teams: 30

Games per team:
home_team
WSH    405
KC     405
PHI    405
NYY    405
NYM    405
MIN    405
MIL    405
SF     405
STL    405
LAA    405
PIT    405
DET    405
CWS    405
COL    405
CIN    405
CHC    405
BOS    405
BAL    405
SEA    404
TEX    404
TOR    404
TB     404
MIA    404
SD     404
ATL    404
LAD    404
HOU    404
CLE    404
AZ     404
ATH    324


## Per-Game League Weather Dataset


In [ ]:
# putting home team after the season
weather_cols = [
    'game_pk', 'game_date', 'season', 'home_team', 'away_team', 'game_start', 'start_hour',
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

league_weather = league[weather_cols].copy()
league_weather = league_weather.sort_values(['game_date', 'home_team']).reset_index(drop=True)

league_weather.to_csv(OUT_LEAGUE_WEATHER, index=False)
print(f"Saved: {os.path.abspath(OUT_LEAGUE_WEATHER)}")
print(f"  {len(league_weather)} games, {league_weather.shape[1]} columns")
print(f"  File size: {os.path.getsize(OUT_LEAGUE_WEATHER) / 1024:.1f} KB")

# Quick validation
print(f"\n--- Validation ---")
print(f"Teams: {league_weather['home_team'].nunique()}")
print(f"Date range: {league_weather['game_date'].min()} to {league_weather['game_date'].max()}")
print(f"Seasons: {sorted(league_weather['season'].unique())}")
print(f"Null counts in weather columns:")
for col in ['temp_f', 'rhum', 'pres', 'wspd_mph']:
    print(f"  {col}: {league_weather[col].isna().sum()}")

Saved: /Users/avabrown/Desktop/DATASCI 192A/Stadium Analysis/Stadium Datasets/Final Datasets/league_weather_2021_2025.csv
  12058 games, 32 columns
  File size: 1868.3 KB

--- Validation ---
Teams: 30
Date range: 2021-04-01 to 2025-09-28
Seasons: [2021, 2022, 2023, 2024, 2025]
Null counts in weather columns:
  temp_f: 3
  rhum: 3
  pres: 3
  wspd_mph: 3


In [6]:
# League-wide weather summary
print("=" * 60)
print("LEAGUE-WIDE WEATHER SUMMARY (2021-2025)")
print("=" * 60)
weather_summary = league_weather[['temp_f', 'rhum', 'pres', 'wspd_mph']].describe()
print(weather_summary.round(2))

print(f"\nPer-season averages:")
season_wx = league_weather.groupby('season')[['temp_f', 'rhum', 'pres', 'wspd_mph']].mean()
print(season_wx.round(2))

LEAGUE-WIDE WEATHER SUMMARY (2021-2025)
         temp_f      rhum      pres  wspd_mph
count  12055.00  12055.00  12055.00  12055.00
mean      71.69     63.16    996.19      7.73
std       12.36     19.68     31.37      3.87
min       20.50      2.30    824.10      0.50
25%       63.90     50.70    990.50      4.90
50%       72.40     65.30   1001.80      7.00
75%       79.90     78.30   1012.90      9.90
max      112.30    100.00   1032.80     35.90

Per-season averages:
        temp_f   rhum    pres  wspd_mph
season                                 
2021     71.18  65.19  996.57      7.96
2022     71.90  62.90  996.36      8.29
2023     71.10  62.75  995.97      7.81
2024     72.01  62.87  995.71      8.16
2025     72.31  62.04  996.33      6.38


## Daily League Average Weather (Seasonal Baseline)

In [ ]:
# daily averages
daily_weather = league_weather.groupby('game_date').agg(
    season=('season', 'first'),
    n_games=('game_pk', 'count'),
    avg_temp_f=('temp_f', 'mean'),
    avg_rhum=('rhum', 'mean'),
    avg_pres=('pres', 'mean'),
    avg_wspd_mph=('wspd_mph', 'mean'),
).reset_index()

for col in ['avg_temp_f', 'avg_rhum', 'avg_pres', 'avg_wspd_mph']:
    daily_weather[col] = daily_weather[col].round(2)

daily_weather = daily_weather.sort_values('game_date').reset_index(drop=True)

daily_weather.to_csv(OUT_DAILY_WEATHER, index=False)
print(f"Saved: {os.path.abspath(OUT_DAILY_WEATHER)}")
print(f"  {len(daily_weather)} unique game dates")
print(f"  File size: {os.path.getsize(OUT_DAILY_WEATHER) / 1024:.1f} KB")

# Validation
print(f"\n--- Validation ---")
print(f"Date range: {daily_weather['game_date'].iloc[0]} to {daily_weather['game_date'].iloc[-1]}")
print(f"Games per day: min={daily_weather['n_games'].min()}, median={daily_weather['n_games'].median():.0f}, max={daily_weather['n_games'].max()}")
print(f"\nDays per season:")
print(daily_weather.groupby('season')['game_date'].count())

Saved: /Users/avabrown/Desktop/DATASCI 192A/Stadium Analysis/Stadium Datasets/Final Datasets/league_daily_weather_2021_2025.csv
  911 unique game dates
  File size: 38.3 KB

--- Validation ---
Date range: 2021-04-01 to 2025-09-28
Games per day: min=1, median=15, max=18

Days per season:
season
2021    182
2022    179
2023    182
2024    185
2025    183
Name: game_date, dtype: int64


In [ ]:
print("--- Daily Weather Baseline (first 15 rows) ---")
daily_weather.head(15)

--- Daily Weather Baseline (first 15 rows) ---


,game_date,season,n_games,avg_temp_f,avg_rhum,avg_pres,avg_wspd_mph
0,2021-04-01,2021,13,51.68,44.49,997.03,9.50
1,2021-04-02,2021,7,53.97,73.61,990.29,8.70
2,2021-04-03,2021,14,57.00,50.00,999.48,9.81
3,2021-04-04,2021,12,64.04,46.77,992.98,8.62
4,2021-04-05,2021,13,57.69,62.35,1005.95,8.53
5,2021-04-06,2021,14,60.29,65.16,994.95,7.87
6,2021-04-07,2021,15,65.45,52.51,995.28,9.49
7,2021-04-08,2021,9,64.44,57.41,976.08,11.37
8,2021-04-09,2021,9,68.90,60.74,993.37,10.70
9,2021-04-10,2021,14,66.22,56.48,990.23,10.56


## League Batting Stats Comparison Table


In [ ]:
master_filtered = master[master['season'].isin(SEASONS)].copy()

for team, year in EXCLUDE:
    master_filtered = master_filtered[~((master_filtered['home_team'] == team) & (master_filtered['season'] == year))]

print(f"Master data for batting stats: {len(master_filtered)} games")
print(f"Teams: {master_filtered['home_team'].nunique()}")

# Stats to compare
STAT_COLS = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks', 'hits', 'hr_h_ratio']
STAT_LABELS = {
    'total_runs': 'Runs/Game',
    'home_runs_hit': 'HR/Game',
    'strikeouts': 'K/Game',
    'walks': 'BB/Game',
    'hits': 'Hits/Game',
    'hr_h_ratio': 'HR:Hits Ratio',
}

Master data for batting stats: 12058 games
Teams: 30


In [ ]:
# only using home games for team averages
team_avgs = master_filtered.groupby('home_team')[STAT_COLS].mean()

league_avg = master_filtered[STAT_COLS].mean()

team_diff = team_avgs.subtract(league_avg)
team_diff.columns = [f'{col}_diff' for col in team_diff.columns]

comparison = team_avgs.copy()
comparison = comparison.join(team_diff)

league_row = league_avg.copy()
for col in STAT_COLS:
    league_row[f'{col}_diff'] = 0.0
comparison.loc['LEAGUE AVG'] = league_row

for col in comparison.columns:
    if 'ratio' in col:
        comparison[col] = comparison[col].round(4)
    else:
        comparison[col] = comparison[col].round(2)

teams_only = comparison.drop('LEAGUE AVG')
teams_sorted = teams_only.sort_values('total_runs', ascending=False)
comparison = pd.concat([teams_sorted, comparison.loc[['LEAGUE AVG']]])

comparison.to_csv(OUT_BATTING_AVGS)
print(f"Saved: {os.path.abspath(OUT_BATTING_AVGS)}")
print(f"  {len(comparison)} rows ({len(comparison) - 1} teams + league average)")
print(f"  File size: {os.path.getsize(OUT_BATTING_AVGS) / 1024:.1f} KB")

Saved: /Users/avabrown/Desktop/DATASCI 192A/Stadium Analysis/Stadium Datasets/Final Datasets/league_batting_avgs_2021_2025.csv
  31 rows (30 teams + league average)
  File size: 2.3 KB


In [ ]:
# Display the full comparison table
print("=" * 90)
print("BATTING STATS COMPARISON: Per-Team Home Game Averages vs League Average (2021-2025)")
print("=" * 90)
print(f"\nNote: All stats are per-game averages (both teams combined) to capture park effects.")
print(f"Diff columns show difference from league average (positive = above average).\n")

# averages column
print("--- Team Averages ---")
display_cols = STAT_COLS
comparison[display_cols]

BATTING STATS COMPARISON: Per-Team Home Game Averages vs League Average (2021-2025)

Note: All stats are per-game averages (both teams combined) to capture park effects.
Diff columns show difference from league average (positive = above average).

--- Team Averages ---


,total_runs,home_runs_hit,strikeouts,walks,hits,hr_h_ratio
home_team,,,,,,
COL,11.29,2.55,15.58,6.35,19.70,0.1288
BOS,9.85,2.23,16.94,6.29,18.15,0.1217
AZ,9.67,2.08,15.96,6.47,17.44,0.1196
CIN,9.61,2.76,17.94,6.81,16.80,0.1637
WSH,9.29,2.33,15.27,6.11,17.28,0.1351
BAL,9.18,2.55,16.60,5.90,17.13,0.1469
LAA,9.18,2.60,18.07,6.65,16.26,0.1610
PHI,9.15,2.52,17.66,6.09,16.66,0.1501
MIN,9.11,2.45,17.50,6.22,16.76,0.1446


In [12]:
# Display differences from league average
print("--- Difference from League Average ---")
diff_cols = [f'{col}_diff' for col in STAT_COLS]
comparison[diff_cols]

--- Difference from League Average ---


,total_runs_diff,home_runs_hit_diff,strikeouts_diff,walks_diff,hits_diff,hr_h_ratio_diff
home_team,,,,,,
COL,2.39,0.24,-1.43,0.03,3.25,-0.0121
BOS,0.95,-0.08,-0.07,-0.03,1.70,-0.0191
AZ,0.77,-0.24,-1.05,0.15,0.99,-0.0213
CIN,0.71,0.44,0.93,0.49,0.35,0.0228
WSH,0.39,0.02,-1.74,-0.21,0.83,-0.0057
BAL,0.28,0.24,-0.40,-0.42,0.68,0.0060
LAA,0.28,0.29,1.07,0.33,-0.19,0.0202
PHI,0.25,0.21,0.65,-0.23,0.21,0.0092
MIN,0.21,0.14,0.49,-0.09,0.31,0.0037
